# T3P Packet Inspector

In [1]:
import struct
import re
import os
import mmap
import numpy as np
from IPython.display import display, HTML

def inspect_and_parse_t3p(filepath, start_packet, end_packet):
    print(f"--- T3P EXACT PACKET INSPECTOR & PARSER ---")
    print(f"File: {os.path.basename(filepath)}")
    print(f"Printing records #{start_packet} through #{end_packet} to Notebook...\n")
    
    # This strict regex guarantees exactly 6 numbers separated by 5 tabs, ending in 10
    hw_trigger_pattern = re.compile(rb'\d+\t\d+\t\d+\t\d+\t\d+\t10\r?\n')
    
    # Lists to accumulate data before NumPy conversion
    photon_list = []
    trigger_list = []
    
    # --- HTML INITIALIZATION ---
    html_rows = []
    
    with open(filepath, 'rb') as f:
        # Use memory-mapping to scan massive files instantly
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Pre-scan for all ASCII HW Trigger injections
        hw_trigger_traps = list(hw_trigger_pattern.finditer(mm))
        
        offset = 0
        record_count = 0
        
        # Global Counters for the entire file
        chip_0_hits = 0  
        chip_1_hits = 0  
        
        # Loop over the ENTIRE file
        while offset < len(mm):
            
            # 1. CHECK FOR ASCII HW TRIGGER INJECTIONS
            if hw_trigger_traps and offset <= hw_trigger_traps[0].start() < offset + 16:
                trap = hw_trigger_traps.pop(0)
                
                # Jump over any corrupted/cut-off bytes directly to the text
                offset = trap.start()
                
                # The text string itself counts as one record in the sequence
                record_count += 1
                
                text_bytes = mm[trap.start() : trap.end()]
                text_str = text_bytes.decode('ascii', errors='ignore').strip()
                
                # Split the string by tabs, convert to integers
                parts = text_str.split('\t')
                if len(parts) == 6:
                    trigger_list.append(tuple(map(int, parts)))
                
                # Format for HTML output if in range
                if start_packet <= record_count <= end_packet:
                    hw_id, mat_idx, toa, tot, trig_id, overflow = parts if len(parts) == 6 else ("-", "-", "-", "-", "-", "-")
                    
                    html_rows.append(f"""
                    <tr class="hw-trigger">
                        <td>{record_count:08d}</td>
                        <td class="highlight-hw">{hw_id}</td>
                        <td>{trap.start():08X}</td>
                        <td>{len(text_bytes)}</td>
                        <td><b>⚡ HW TRIGGER</b></td>
                        <td>{mat_idx}</td>
                        <td>{toa}</td>
                        <td>{tot}</td>
                        <td>{trig_id}</td>
                        <td>{overflow}</td>
                    </tr>
                    """)
                
                # Resume standard 16-byte reading immediately after the text ends
                offset = trap.end()
                continue
                
            # 2. READ THE STANDARD 16-BYTE BINARY PACKET (PHOTON HITS)
            if offset + 16 > len(mm):
                break 
                
            packet = mm[offset : offset + 16]
            
            # Unpack all 5 variables from the C-Struct
            matrixIdx, toa, overflow, ftoa, tot = struct.unpack('<IQBBH', packet)
            record_count += 1
            
            # Store the photon hit as a tuple
            photon_list.append((matrixIdx, toa, overflow, ftoa, tot))
            
            # Global Tally logic
            if overflow == 0:
                chip_0_hits += 1
            else:
                chip_1_hits += 1
            
            # 3. FORMAT FOR HTML OUTPUT IF IN RANGE
            if start_packet <= record_count <= end_packet:
                html_rows.append(f"""
                <tr class="photon-hit">
                    <td>{record_count:08d}</td>
                    <td class="dim">-</td>
                    <td>{offset:08X}</td>
                    <td>16</td>
                    <td>🔵 PHOTON HIT</td>
                    <td>{matrixIdx}</td>
                    <td>{toa}</td>
                    <td>{tot}</td>
                    <td>{ftoa}</td>
                    <td>{overflow}</td>
                </tr>
                """)
                
            offset += 16
            
        mm.close()
    
    # ==========================================
    # HTML GENERATION & NOTEBOOK DISPLAY
    # ==========================================
    html_content = f"""
    <style>
        .t3p-table-container {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #333; }}
        .t3p-table-container h2 {{ border-bottom: 2px solid #2c3e50; padding-bottom: 10px; color: #2c3e50; margin-top: 0; }}
        .t3p-summary {{ background: #f8f9fa; padding: 15px; border-radius: 8px; border: 1px solid #e9ecef; margin-bottom: 20px; }}
        .t3p-table {{ border-collapse: collapse; width: 100%; background-color: #fff; box-shadow: 0 1px 3px rgba(0,0,0,0.1); font-family: monospace; font-size: 13px; }}
        .t3p-table th, .t3p-table td {{ border: 1px solid #e9ecef; padding: 10px 12px; text-align: center; }}
        .t3p-table th {{ background-color: #2c3e50 !important; color: white !important; text-transform: uppercase; letter-spacing: 0.05em; font-size: 12px; }}
        .t3p-table tr:hover {{ background-color: #f1f3f5; }}
        .t3p-table .hw-trigger {{ background-color: #fff4e5; }}
        .t3p-table .photon-hit {{ background-color: #ffffff; }} 
        .t3p-table .highlight-hw {{ font-weight: bold; color: #d35400; }}
        .t3p-table .dim {{ color: #adb5bd; }}
    </style>
    
    <div class="t3p-table-container">
        <h2>T3P Packet Inspection Report</h2>
        <div class="t3p-summary">
            <strong>File:</strong> {os.path.basename(filepath)}<br>
            <strong>Records Displayed:</strong> #{start_packet} to #{end_packet}
        </div>
        <table class="t3p-table">
            <thead>
                <tr>
                    <th>RECORD #</th>
                    <th title="Only applies to HW Triggers">HW ID</th>
                    <th>BYTE OFFSET</th>
                    <th>LENGTH</th>
                    <th>PACKET TYPE</th>
                    <th>matrixIdx</th>
                    <th>ToA</th>
                    <th>ToT</th>
                    <th title="fToA for Photons / Trigger ID for HW Triggers">fToA / Trig ID</th>
                    <th>Overflow</th>
                </tr>
            </thead>
            <tbody>
                {"".join(html_rows)}
            </tbody>
        </table>
    </div>
    """
    
    # Render the HTML directly in the Jupyter Notebook cell output
    display(HTML(html_content))

    # ==========================================
    # NUMPY ARRAY CONVERSION
    # ==========================================
    photon_dtype = np.dtype([
        ('matrixIdx', np.uint32),
        ('toa', np.uint64),
        ('overflow', np.uint8),
        ('ftoa', np.uint8),
        ('tot', np.uint16)
    ])
    
    trigger_dtype = np.dtype([
        ('record_idx', np.uint32),
        ('matrixIdx',  np.uint32),
        ('toa',        np.uint64),
        ('tot',        np.uint32),
        ('trigger_id', np.uint32), 
        ('overflow',   np.uint32)
    ])

    photon_array = np.array(photon_list, dtype=photon_dtype)
    trigger_array = np.array(trigger_list, dtype=trigger_dtype)
        
    print("\n--- INSPECTION COMPLETE ---")
    print(f"► Total HW triggers (Text):           {len(trigger_array)}")
    print(f"► Total photon hits (Binary):         {len(photon_array)}")
    print(f"  ├─ Hits on Chip 0 (overflow == 0):  {chip_0_hits}")
    print(f"  └─ Hits on Chip 1+ (overflow != 0): {chip_1_hits}")
    
    print("\n--- MATRIX INDEX ANALYSIS ---")
    if len(photon_array) > 0:
        max_idx = np.max(photon_array['matrixIdx'])
        print(f"► Maximum matrixIdx found: {max_idx}")
        
        if max_idx > 65535:
            print("  └─ Conclusion: Index exceeds 65,535. The camera uses a GLOBAL indexing scheme across both chips (e.g., 256x512).")
        else:
            print("  └─ Conclusion: Index is <= 65,535. The camera uses SEPARATE indexing per chip (resets to 0 for the second chip).")
    else:
        print("► No photon hits found to analyze.")

    print("\n--- OVERFLOW ANALYSIS ---")  
    if len(photon_array) > 0:
        max_overflow = np.max(photon_array['overflow'])
        print(f"► Maximum overflow found: {max_overflow}")

    if len(trigger_array) > 0:
        max_idx = np.max(trigger_array['record_idx'])
        print(f"\n► Triggers Array Length: {len(trigger_array)}")
        if len(trigger_array) == max_idx + 1:
            print("  └─ Status: PERFECT MATCH! No fake triggers were captured.")
        else:
            print("  └─ Status: WARNING! The array length doesn't match the final index. Some triggers are missing or skipped.")    

    return photon_array, trigger_array

# ==========================================
# EXECUTION
# ==========================================
t3p_file = r"D:\Test_Meas_25.6\meas2\sync02_25.6_r2.t3p"

start = 0
end = 5

# Capture the returned arrays
photons, triggers = inspect_and_parse_t3p(t3p_file, start, end)

--- T3P EXACT PACKET INSPECTOR & PARSER ---
File: sync02_25.6_r2.t3p
Printing records #0 through #5 to Notebook...



RECORD #,HW ID,BYTE OFFSET,LENGTH,PACKET TYPE,matrixIdx,ToA,ToT,fToA / Trig ID,Overflow
00000001,0,00000000,14,⚡ HW TRIGGER,0,62,0,0,10
00000002,1,0000000E,18,⚡ HW TRIGGER,0,62,0,49152,10
00000003,-,00000020,16,🔵 PHOTON HIT,107672,50155,7,17,1
00000004,-,00000030,16,🔵 PHOTON HIT,107415,50154,15,5,1
00000005,-,00000040,16,🔵 PHOTON HIT,107416,50154,73,7,1



--- INSPECTION COMPLETE ---
► Total HW triggers (Text):           1204
► Total photon hits (Binary):         941279
  ├─ Hits on Chip 0 (overflow == 0):  460031
  └─ Hits on Chip 1+ (overflow != 0): 481248

--- MATRIX INDEX ANALYSIS ---
► Maximum matrixIdx found: 131071
  └─ Conclusion: Index exceeds 65,535. The camera uses a GLOBAL indexing scheme across both chips (e.g., 256x512).

--- OVERFLOW ANALYSIS ---
► Maximum overflow found: 1

► Triggers Array Length: 1204
  └─ Status: PERFECT MATCH! No fake triggers were captured.


## T3P inspection with html table

In [ ]:
import struct
import re
import os
import mmap
import numpy as np
import webbrowser

def inspect_and_parse_t3p(filepath, start_packet, end_packet):
    print(f"--- T3P EXACT PACKET INSPECTOR & PARSER ---")
    print(f"File: {os.path.basename(filepath)}")
    print(f"Printing records #{start_packet} through #{end_packet} to HTML...")
    print(f"Scanning the entire file to build NumPy arrays...\n")
    
    # This strict regex guarantees exactly 6 numbers separated by 5 tabs, ending in 10
    hw_trigger_pattern = re.compile(rb'\d+\t\d+\t\d+\t\d+\t\d+\t10\r?\n')
    
    # Lists to accumulate data before NumPy conversion
    photon_list = []
    trigger_list = []
    
    # --- HTML INITIALIZATION ---
    html_rows = []
    
    with open(filepath, 'rb') as f:
        # Use memory-mapping to scan massive files instantly
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Pre-scan for all ASCII HW Trigger injections
        hw_trigger_traps = list(hw_trigger_pattern.finditer(mm))
        
        offset = 0
        record_count = 0
        
        # Global Counters for the entire file
        chip_0_hits = 0  
        chip_1_hits = 0  
        
        # Loop over the ENTIRE file
        while offset < len(mm):
            
            # 1. CHECK FOR ASCII HW TRIGGER INJECTIONS
            if hw_trigger_traps and offset <= hw_trigger_traps[0].start() < offset + 16:
                trap = hw_trigger_traps.pop(0)
                
                # Jump over any corrupted/cut-off bytes directly to the text
                offset = trap.start()
                
                # The text string itself counts as one record in the sequence
                record_count += 1
                
                text_bytes = mm[trap.start() : trap.end()]
                text_str = text_bytes.decode('ascii', errors='ignore').strip()
                
                # Split the string by tabs, convert to integers
                parts = text_str.split('\t')
                if len(parts) == 6:
                    trigger_list.append(tuple(map(int, parts)))
                
                # Format for HTML output if in range
                if start_packet <= record_count <= end_packet:
                    hw_id, mat_idx, toa, tot, trig_id, overflow = parts if len(parts) == 6 else ("-", "-", "-", "-", "-", "-")
                    
                    html_rows.append(f"""
                    <tr class="hw-trigger">
                        <td>{record_count:08d}</td>
                        <td class="highlight-hw">{hw_id}</td>
                        <td>{trap.start():08X}</td>
                        <td>{len(text_bytes)}</td>
                        <td><b>⚡ HW TRIGGER</b></td>
                        <td>{mat_idx}</td>
                        <td>{toa}</td>
                        <td>{tot}</td>
                        <td>{trig_id}</td>
                        <td>{overflow}</td>
                    </tr>
                    """)
                
                # Resume standard 16-byte reading immediately after the text ends
                offset = trap.end()
                continue
                
            # 2. READ THE STANDARD 16-BYTE BINARY PACKET (PHOTON HITS)
            if offset + 16 > len(mm):
                break 
                
            packet = mm[offset : offset + 16]
            
            # Unpack all 5 variables from the C-Struct
            matrixIdx, toa, overflow, ftoa, tot = struct.unpack('<IQBBH', packet)
            record_count += 1
            
            # Store the photon hit as a tuple
            photon_list.append((matrixIdx, toa, overflow, ftoa, tot))
            
            # Global Tally logic
            if overflow == 0:
                chip_0_hits += 1
            else:
                chip_1_hits += 1
            
            # 3. FORMAT FOR HTML OUTPUT IF IN RANGE
            if start_packet <= record_count <= end_packet:
                html_rows.append(f"""
                <tr class="photon-hit">
                    <td>{record_count:08d}</td>
                    <td class="dim">-</td>
                    <td>{offset:08X}</td>
                    <td>16</td>
                    <td>🔵 PHOTON HIT</td>
                    <td>{matrixIdx}</td>
                    <td>{toa}</td>
                    <td>{tot}</td>
                    <td>{ftoa}</td>
                    <td>{overflow}</td>
                </tr>
                """)
                
            offset += 16
            
        mm.close()
    
    # ==========================================
    # HTML GENERATION
    # ==========================================
    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>T3P Packet Inspection</title>
        <style>
            body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f8f9fa; padding: 20px; color: #333; }}
            h2 {{ border-bottom: 2px solid #2c3e50; padding-bottom: 10px; color: #2c3e50; }}
            .summary {{ background: #fff; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-bottom: 20px; }}
            table {{ border-collapse: collapse; width: 100%; background-color: #fff; box-shadow: 0 2px 8px rgba(0,0,0,0.1); font-family: monospace; font-size: 14px; }}
            th, td {{ border: 1px solid #e9ecef; padding: 12px 15px; text-align: center; }}
            th {{ background-color: #2c3e50; color: white; text-transform: uppercase; letter-spacing: 0.05em; font-size: 13px; }}
            tr:hover {{ background-color: #f1f3f5; }}
            .hw-trigger {{ background-color: #fff4e5; }} /* Light orange tint */
            .photon-hit {{ background-color: #ffffff; }} 
            .highlight-hw {{ font-weight: bold; color: #d35400; }}
            .dim {{ color: #adb5bd; }}
        </style>
    </head>
    <body>
        <h2>T3P Packet Inspection Report</h2>
        <div class="summary">
            <strong>File:</strong> {os.path.basename(filepath)}<br>
            <strong>Records Displayed:</strong> #{start_packet} to #{end_packet}
        </div>
        <table>
            <thead>
                <tr>
                    <th>RECORD #</th>
                    <th title="Only applies to HW Triggers">HW ID</th>
                    <th>BYTE OFFSET</th>
                    <th>LENGTH</th>
                    <th>PACKET TYPE</th>
                    <th>matrixIdx</th>
                    <th>ToA</th>
                    <th>ToT</th>
                    <th title="fToA for Photons / Trigger ID for HW Triggers">fToA / Trig ID</th>
                    <th>Overflow</th>
                </tr>
            </thead>
            <tbody>
                {"".join(html_rows)}
            </tbody>
        </table>
    </body>
    </html>
    """
    
    # Save the HTML to a file in the same directory as the .t3p file
    html_filepath = os.path.join(os.path.dirname(filepath), "inspection_report.html")
    with open(html_filepath, 'w', encoding='utf-8') as html_file:
        html_file.write(html_content)
        
    print(f"✅ HTML report generated successfully: {html_filepath}")
    webbrowser.open('file://' + os.path.realpath(html_filepath))

    # ==========================================
    # NUMPY ARRAY CONVERSION
    # ==========================================
    photon_dtype = np.dtype([
        ('matrixIdx', np.uint32),
        ('toa', np.uint64),
        ('overflow', np.uint8),
        ('ftoa', np.uint8),
        ('tot', np.uint16)
    ])
    
    trigger_dtype = np.dtype([
        ('record_idx', np.uint32),
        ('matrixIdx',  np.uint32),
        ('toa',        np.uint64),
        ('tot',        np.uint32),
        ('trigger_id', np.uint32), 
        ('overflow',   np.uint32)
    ])

    photon_array = np.array(photon_list, dtype=photon_dtype)
    trigger_array = np.array(trigger_list, dtype=trigger_dtype)
        
    print("\n--- INSPECTION COMPLETE ---")
    print(f"► Total HW triggers (Text):           {len(trigger_array)}")
    print(f"► Total photon hits (Binary):         {len(photon_array)}")
    print(f"  ├─ Hits on Chip 0 (overflow == 0):  {chip_0_hits}")
    print(f"  └─ Hits on Chip 1+ (overflow != 0): {chip_1_hits}")
    
    print("\n--- MATRIX INDEX ANALYSIS ---")
    if len(photon_array) > 0:
        max_idx = np.max(photon_array['matrixIdx'])
        print(f"► Maximum matrixIdx found: {max_idx}")
        
        # 256 * 256 = 65,536 pixels per chip. Indices run 0 to 65,535.
        if max_idx > 65535:
            print("  └─ Conclusion: Index exceeds 65,535. The camera uses a GLOBAL indexing scheme across both chips (e.g., 256x512).")
        else:
            print("  └─ Conclusion: Index is <= 65,535. The camera uses SEPARATE indexing per chip (resets to 0 for the second chip).")
    else:
        print("► No photon hits found to analyze.")

    print("\n--- OVERFLOW ANALYSIS ---")  
    if len(photon_array) > 0:
        max_overflow = np.max(photon_array['overflow'])
        print(f"► Maximum overflow found: {max_overflow}")

    if len(trigger_array) > 0:
        max_idx = np.max(trigger_array['record_idx'])
        print(f"\n► Triggers Array Length: {len(trigger_array)}")
        if len(trigger_array) == max_idx + 1:
            print("  └─ Status: PERFECT MATCH! No fake triggers were captured.")
        else:
            print("  └─ Status: WARNING! The array length doesn't match the final index. Some triggers are missing or skipped.")    

    return photon_array, trigger_array

# ==========================================
# EXECUTION
# ==========================================
t3p_file = r"G:\האחסון שלי\X-Ray-IFM\Test Files\Sync_test\sync_test_25.5_r0.t3p"

start = 0
end = 5

# Capture the returned arrays
photons, triggers = inspect_and_parse_t3p(t3p_file, start, end)

# .h5 Bins Inspector

In [ ]:
import h5py
import numpy as np
import os

def inspect_px5_h5(filepath, start_bin, end_bin):
    print(f"--- HDF5 PX5 DATA INSPECTOR ---")
    print(f"File: {os.path.basename(filepath)}")

    try:
        with h5py.File(filepath, 'r') as f:
            # 1. Read the Metadata attributes written by the DAQ script
            bin_s = f.attrs['bin_s']
            
            # 2. Access the main dataset
            if 'px5CountsPerBin' not in f:
                print("Error: Dataset 'px5CountsPerBin' not found in file.")
                return
                
            dset = f['px5CountsPerBin']
            total_bins = dset.shape[0]
            
            print(f"Total Bins   : {total_bins:,}")
            print(f"Bin Resolution: {bin_s * 1e6:.1f} µs ({bin_s} s)")
            print(f"File Duration: {total_bins * bin_s:.2f} seconds")
            
            # We can sum the entire array instantly to find the total hits
            print(f"Total Photons: {np.sum(dset):,}") 
            print("-" * 55)
            
            # 3. Validate user input range
            if start_bin < 0: start_bin = 0
            if end_bin >= total_bins: end_bin = total_bins - 1
            if start_bin > end_bin:
                print("Invalid range selected.")
                return
            
            print(f"Scanning bins #{start_bin} to #{end_bin}...\n")
            print(f"{'BIN INDEX':<12} | {'TIME (Seconds)':<15} | {'PHOTON COUNT'}")
            print("-" * 45)
            
            # 4. Extract only the specific slice of memory requested
            counts_in_range = dset[start_bin : end_bin + 1]
            
            for i, count in enumerate(counts_in_range):
                actual_bin = start_bin + i
                time_s = actual_bin * bin_s
                
                # Add a visual flag if a photon was actually detected in this bin
                if count > 0:
                    count_str = f"{count}  <-- 🟢 HIT"
                else:
                    count_str = str(count)
                    
                print(f"[{actual_bin:08d}]   | {time_s:<15.6f} | {count_str}")
                
    except Exception as e:
        print(f"Failed to read HDF5 file: {e}")

# ==========================================
# EXECUTION
# ==========================================
# Point this to your generated DAQ file
h5_file = r"G:\האחסון שלי\X-Ray-IFM\Test Files\Short_Test\short_test_23.6_000.h5"

# Select the range of bins you want to look at
start = 0
end   = 50

inspect_px5_h5(h5_file, start, end)